# MedGemma 1.5 4B: smoke test and resolution gate

Two questions, answered before any GPU quota is committed to a sweep:

1. Does the model run on free Kaggle hardware, and how fast?
2. **Does the 896x896 encoding destroy the small print on a full-page A4 report?**

Question 2 decides the shape of the project. MedGemma encodes every image at
896x896 into 256 tokens. A 1588x2246 render is downsampled hard before the
model sees anything, so a `scale` degradation axis could end up measuring a
resize function rather than the model. Four conditions on the same image tell
us which:

| Condition | What it isolates |
|---|---|
| `squash` | The naive baseline |
| `pan_and_scan` | Does tiling recover the small text? |
| `crop_table` | Is the limit resolution, or the model itself? |
| `int4` | Does quantisation cost accuracy on top of that? |

**Setup:** Accelerator `GPU T4 x2`. Attach the dataset of generated reports.
Add `HF_TOKEN` under Add-ons > Secrets.

In [ ]:
!pip install -q -U "transformers>=4.50" accelerate bitsandbytes

In [ ]:
import os, json, time, glob, gc
import torch
from PIL import Image

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print(torch.cuda.get_device_name(0))
cap = torch.cuda.get_device_capability(0)
print("compute capability:", cap)
# bf16 needs capability >= 8.0. T4 is 7.5 and P100 is 6.0, so both fall back
# to fp16. Google's sample code uses bf16 and will crash or crawl here.
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
print("using dtype:", DTYPE)

In [ ]:
# Recursive, because how deep Kaggle extracts depends on how the zip was built.
# If nothing is found, the tree below distinguishes "dataset not attached" from
# "attached but nested somewhere unexpected", which a bare glob cannot.
ROOT = "/kaggle/input"

if not os.path.isdir(ROOT) or not os.listdir(ROOT):
    raise SystemExit(
        "Nothing under /kaggle/input, so no dataset is attached.\n"
        "Right-hand panel > Input > Add Input > Datasets > add yours.\n"
        "A dataset still processing after upload also shows up empty here."
    )

print("attached under /kaggle/input:")
for dirpath, dirnames, filenames in os.walk(ROOT):
    depth = dirpath.count(os.sep) - ROOT.count(os.sep)
    print("  " * depth + os.path.basename(dirpath) + "/")
    for f in sorted(filenames)[:4]:
        print("  " * (depth + 1) + f)
    if len(filenames) > 4:
        print("  " * (depth + 1) + f"... {len(filenames) - 4} more")

DATA = sorted(glob.glob(f"{ROOT}/**/*.png", recursive=True))
print(f"\n{len(DATA)} images found")
if not DATA:
    raise SystemExit(
        "Dataset is attached but contains no PNGs. Check that the upload held\n"
        "the images themselves, and that Kaggle finished unzipping it."
    )

IMG_PATH = DATA[0]
GT_PATH = IMG_PATH.replace(".png", ".json")
if not os.path.exists(GT_PATH):
    raise SystemExit(f"Found {IMG_PATH} but no ground truth beside it at {GT_PATH}")

gt = json.load(open(GT_PATH))
img = Image.open(IMG_PATH).convert("RGB")
print("using:", IMG_PATH)
print("image size:", img.size)
print("ground truth tests:", len(gt["tests"]))

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-1.5-4b-it"

t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,          # 'dtype', not the deprecated 'torch_dtype'
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
load_s = time.time() - t0
print(f"loaded in {load_s:.1f}s")
print(f"weights VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

In [ ]:
PROMPT = """Extract all laboratory test results from this report as JSON.

Output ONLY valid JSON matching this schema, with no explanation, no markdown
fences, and no additional text:

{
  "patient": {"name": "", "age": null, "sex": ""},
  "tests": [
    {"test_name": "", "value": null, "unit": "",
     "ref_low": null, "ref_high": null, "ref_text": "", "flag": ""}
  ]
}

Rules:
- Copy test names exactly as printed on the report.
- If a value is qualitative (e.g. "Negative"), put it in "value" as a string.
- If a reference range is text rather than numeric, use "ref_text".
- Do not infer or invent any value that is not printed on the report.
"""


def extract(image, pan_and_scan=False, max_new_tokens=1200):
    """Run one extraction and return (text, seconds, peak_gb)."""
    # Set on the image processor directly. This is the reliable place for it;
    # kwargs do not always survive the trip through apply_chat_template.
    processor.image_processor.do_pan_and_scan = pan_and_scan

    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ]}]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=DTYPE)

    n_image_tokens = int((inputs["input_ids"] == processor.tokenizer.convert_tokens_to_ids("<image_soft_token>")).sum()) if "<image_soft_token>" in processor.tokenizer.get_vocab() else -1
    input_len = inputs["input_ids"].shape[-1]

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    secs = time.time() - t0

    text = processor.decode(out[0][input_len:], skip_special_tokens=True)
    peak = torch.cuda.max_memory_allocated() / 1e9
    return text, secs, peak, input_len, n_image_tokens

In [ ]:
import re

def parse_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e == -1:
        return None, "no_json_found"
    try:
        return json.loads(text[s:e + 1]), None
    except json.JSONDecodeError as err:
        return None, f"parse_error: {err}"


def score(pred, gt):
    """Crude value-recall against ground truth. Not the real harness, just
    enough signal to tell whether the model read the page at all."""
    if not pred or "tests" not in pred:
        return 0.0, 0
    want = {f"{t['value']:.{t['decimals']}f}" for t in gt["tests"]}
    got = set()
    for t in pred.get("tests", []):
        v = t.get("value")
        if v is None:
            continue
        try:
            got.update({f"{float(v):.{d}f}" for d in range(4)})
        except (TypeError, ValueError):
            got.add(str(v))
    return len(want & got) / len(want), len(pred.get("tests", []))

In [ ]:
# Crop to the results table. The generator puts it in the top ~45% of the page,
# so this isolates 'the model cannot read small text' from 'the model cannot
# find the table'.
w, h = img.size
img_crop = img.crop((0, 0, w, int(h * 0.45)))
print("crop size:", img_crop.size)

results = {}
for name, image, pas in [
    ("squash", img, False),
    ("pan_and_scan", img, True),
    ("crop_table", img_crop, False),
]:
    text, secs, peak, in_len, n_img = extract(image, pan_and_scan=pas)
    pred, err = parse_json(text)
    recall, n_tests = score(pred, gt)
    results[name] = dict(seconds=round(secs, 1), peak_gb=round(peak, 2),
                         input_tokens=in_len, parse_error=err,
                         tests_returned=n_tests, value_recall=round(recall, 3))
    print(f"{name:14s} {secs:6.1f}s  {peak:5.2f}GB  in={in_len:5d}  "
          f"tests={n_tests:2d}  recall={recall:.2f}  {err or ''}")
    print(text[:300].replace("\n", " "), "\n")

In [ ]:
# 4-bit. Reloads the model, so run this after the fp16 numbers are recorded.
del model
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)
t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto")
print(f"4-bit loaded in {time.time()-t0:.1f}s, "
      f"VRAM {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

best = "pan_and_scan" if results["pan_and_scan"]["value_recall"] >= results["squash"]["value_recall"] else "squash"
text, secs, peak, in_len, _ = extract(img, pan_and_scan=(best == "pan_and_scan"))
pred, err = parse_json(text)
recall, n_tests = score(pred, gt)
results["int4"] = dict(seconds=round(secs, 1), peak_gb=round(peak, 2),
                       input_tokens=in_len, parse_error=err,
                       tests_returned=n_tests, value_recall=round(recall, 3),
                       matched_condition=best)
print(f"int4 ({best}): {secs:.1f}s  {peak:.2f}GB  tests={n_tests}  recall={recall:.2f}")

In [ ]:
summary = {"model": MODEL_ID, "gpu": torch.cuda.get_device_name(0),
           "dtype": str(DTYPE), "load_seconds": round(load_s, 1),
           "image": os.path.basename(IMG_PATH), "image_size": list(img.size),
           "gt_test_count": len(gt["tests"]), "conditions": results}

print(json.dumps(summary, indent=2))
with open("/kaggle/working/smoke_test.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nDECISION")
print("  If pan_and_scan recall clearly beats squash, keep tiling on and accept")
print("  the slower per-image cost, then cut the sweep to 5 axes.")
print("  If crop_table beats both by a wide margin, the vision encoder is the")
print("  bottleneck: enlarge the type in the templates before building 7 more.")
print("  If squash is already fine, build the remaining templates unchanged.")